# 🎓 FestNest AI AutoFill — Phase 5: LoRA Fine-Tuning
### Model: `Qwen/Qwen2.5-1.5B-Instruct` | Method: 4-bit QLoRA (PEFT)
### Hardware Target: Free Google Colab T4 GPU (< 15-20 minutes)

This notebook fine-tunes **Qwen2.5-1.5B-Instruct** on the FestNest event poster dataset prepared in Phase 4.

#### Target Output Schema:
```json
{
  "event_name": "",
  "event_type": "", // Exactly one of: Hackathon, Cultural Fest, Workshop, Competition, Tech Talk, Sports, Mega Fest
  "date": "",
  "time": "",
  "venue": "",
  "organizer": "",
  "prize_pool": "",
  "registration_deadline": "",
  "contact": "",
  "description": ""
}
```

> **Colab Hardware Setup**:
> Go to **Runtime** → **Change runtime type** → select **T4 GPU** → click **Save** before running.

## Step 1: Verify GPU Environment

In [ ]:
# Check NVIDIA GPU allocation
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    raise RuntimeError("GPU not detected! Please go to Runtime -> Change runtime type -> T4 GPU.")

## Step 2: Install Required Libraries

In [ ]:
!pip install -q -U transformers peft bitsandbytes trl datasets accelerate "huggingface-hub<2.0"

## Step 3: Clone Repository & Verify Dataset

In [ ]:
import os
import json
from pathlib import Path

# If inside cloned repo, navigate to project root
if os.path.exists('festnest-ai-autofill'):
    %cd festnest-ai-autofill

os.makedirs('data/dataset', exist_ok=True)

# Check if dataset is already present
if not os.path.exists('data/dataset/train.jsonl'):
    print('data/dataset/train.jsonl not found.')
    print('Please upload train.jsonl and validation.jsonl using the upload widget below:')
    try:
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith('.jsonl'):
                os.replace(fname, f'data/dataset/{fname}')
    except Exception as e:
        print(f'Upload widget notice: {e}')

# If validation.jsonl is missing, check if user wants to upload it or auto-create it from train.jsonl
if os.path.exists('data/dataset/train.jsonl') and not os.path.exists('data/dataset/validation.jsonl'):
    print('validation.jsonl is missing.')
    try:
        print('You can upload validation.jsonl now (or cancel to auto-generate from train.jsonl):')
        from google.colab import files
        uploaded = files.upload()
        for fname in uploaded.keys():
            if 'val' in fname.lower() or fname.endswith('.jsonl'):
                os.replace(fname, 'data/dataset/validation.jsonl')
    except Exception as e:
        pass

# Automatic fallback: if validation.jsonl is still missing, split the last 2 records from train.jsonl
if os.path.exists('data/dataset/train.jsonl') and not os.path.exists('data/dataset/validation.jsonl'):
    print('Auto-generating validation.jsonl from train.jsonl...')
    with open('data/dataset/train.jsonl', 'r', encoding='utf-8') as f:
        lines = [l.strip() for l in f if l.strip()]
    if len(lines) > 2:
        val_lines = lines[-2:]
        train_lines = lines[:-2]
        with open('data/dataset/validation.jsonl', 'w', encoding='utf-8') as f:
            f.write('\n'.join(val_lines) + '\n')
        with open('data/dataset/train.jsonl', 'w', encoding='utf-8') as f:
            f.write('\n'.join(train_lines) + '\n')
        print(f'Validation split created ({len(val_lines)} samples, Train: {len(train_lines)} samples).')
    else:
        with open('data/dataset/validation.jsonl', 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines) + '\n')
        print('Validation split created from train.jsonl.')

assert os.path.exists('data/dataset/train.jsonl'), 'data/dataset/train.jsonl missing!'
assert os.path.exists('data/dataset/validation.jsonl'), 'data/dataset/validation.jsonl missing!'
print('Dataset files verified successfully!')
print(f"Train samples: {sum(1 for _ in open('data/dataset/train.jsonl', encoding='utf-8'))}")
print(f"Validation samples: {sum(1 for _ in open('data/dataset/validation.jsonl', encoding='utf-8'))}")

## Step 4: Load Tokenizer & 4-bit Quantized Base Model (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 2. Configure 4-bit NormalFloat4 (NF4) quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 3. Load Base Model into GPU with float16 for T4 compatibility
print(f"Loading base model {MODEL_ID} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

# Ensure all non-quantized parameters are in Float16 (fixes T4 BFloat16 conflict)
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

print("Base model successfully loaded into VRAM!")

## Step 5: Configure LoRA Adapters (PEFT)

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## Step 6: Load & Preprocess Datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset('json', data_files={
    'train': 'data/dataset/train.jsonl',
    'validation': 'data/dataset/validation.jsonl',
})
print(dataset)

def format_example(example):
    if 'text' in example and example['text']:
        return {'text': example['text']}
    return {
        'text': tokenizer.apply_chat_template(
            example['messages'],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_dataset = dataset['train'].map(format_example)
eval_dataset = dataset['validation'].map(format_example)

print('Sample ChatML Prompt Format Preview:')
print(train_dataset[0]['text'][:350] + '\n...')

## Step 7: Train with TRL `SFTTrainer`

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_ADAPTER_DIR = "models/qwen2.5-1.5b-festnest-lora"
CHECKPOINT_DIR = "models/checkpoints"

# Ensure any lingering BFloat16 weights are cast to Float16 for T4 GPU compatibility
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

sft_kwargs = {
    "output_dir": CHECKPOINT_DIR,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 5,
    "fp16": True,
    "logging_steps": 2,
    "save_strategy": "epoch",
    "eval_strategy": "epoch",
    "save_total_limit": 2,
    "dataset_text_field": "text",
    "report_to": "none",
}

if hasattr(SFTConfig, "max_length"):
    sft_kwargs["max_length"] = 1536
else:
    sft_kwargs["max_seq_length"] = 1536

training_args = SFTConfig(**sft_kwargs)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Launching fine-tuning on T4 GPU...")
train_result = trainer.train()
print("Training finished successfully!")

## Step 8: Save LoRA Adapter & Metadata

In [ ]:
import json
from datetime import datetime
from pathlib import Path

out_dir = Path(OUTPUT_ADAPTER_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving trained LoRA adapter to {out_dir}...")
trainer.model.save_pretrained(str(out_dir))
tokenizer.save_pretrained(str(out_dir))

meta = {
    "timestamp": datetime.now().isoformat(),
    "base_model": MODEL_ID,
    "hyperparameters": {
        "epochs": 3,
        "batch_size": 2,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "lora_r": 16,
        "lora_alpha": 32,
        "quantization": "4bit-nf4",
    },
    "metrics": train_result.metrics,
}
with open(out_dir / "training_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved adapter files:")
!ls -lh {OUTPUT_ADAPTER_DIR}

## Step 9: Inference Verification on Held-Out Test Split

In [ ]:
import json

# Load first test sample from held-out test split
with open("data/dataset/test.jsonl", "r", encoding="utf-8") as f:
    test_sample = json.loads(f.readline())

test_ocr = test_sample["input"]
print("--- Test Input OCR Snippet ---")
print(test_ocr[:200] + "...\n")

prompt_messages = [
    {
        "role": "system",
        "content": "You are an AI assistant that extracts structured event information from poster OCR text into JSON format matching FestNest schemas.",
    },
    {
        "role": "user",
        "content": f"Extract event details as JSON from this OCR text:\n\n{test_ocr}",
    },
]

prompt_text = tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=== Fine-Tuned Model Prediction ===")
print(generated_text)
print("\n=== Ground Truth JSON ===")
print(test_sample["output"])

## Step 10: Zip Adapter Archive for Download
Download the fine-tuned LoRA adapter to your local machine so Phase 6 evaluation can run against it.

In [ ]:
import shutil
from google.colab import files

archive_name = "qwen2.5-1.5b-festnest-lora"
shutil.make_archive(archive_name, "zip", OUTPUT_ADAPTER_DIR)
print(f"Created archive: {archive_name}.zip")

# Trigger browser download
files.download(f"{archive_name}.zip")
print("Download initiated! Extract this zip into your local models/qwen2.5-1.5b-festnest-lora directory.")